# Part E: Analysis
Answer in the design document, grounding your answers in your own results where applicable.

1. **Coordinate parameterization**. What did you regress in Part B, and what are its failure modes? Discuss heat-map and direct-coordinate formulations and the trade-offs between them: what each does under occlusion, what each costs at your budget, and what the budget would have had to be for you to choose the other one.

2. **Normalization and metric conditioning**. Why does the choice of NME normalization change the shape of your Part C curves? Consider what happens to inter-ocular normalization as yaw increases, and what that implies about comparing published landmark numbers across papers.

3. **Confidence and error**. Part D needs a predictor of error, and nothing available at inference time measures error directly. Using your own results, characterize the gap between whatever signal you used and the error itself. Where does the proxy hold, and where does it fail in each direction?

4. **Training distribution**. In-cabin occlusion has a specific structure that web-collected data does not: hands, wheel, seat-belt, sun visor, IR illumination. What does this imply about which of your results transfer to deployment, and which are artifacts of the protocol?

이 노트북은 서술형 답 자체가 아니라, 그 답을 뒷받침하는 **수치 근거**의
재현. 전체 서술은 `DESIGN.md` §6 참조.

## 1. 준비

In [2]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [3]:
from src.shared.nme_metrics import nme_batch
from src.shared.yaw_estimation import yaw_from_landmarks, bin_by_yaw
from src.application.accept_reject_use_case import disagreement_signal
from src.models.landmark_net import LandmarkNet, count_params_mb

import numpy as np

In [8]:
REPO_ROOT = os.path.abspath("../..")
yaw_bins = [0, 10, 20, 30, 45, 90]

In [5]:
preds_result = np.load(f"{REPO_ROOT}/output/results/preds.npz", allow_pickle="True")
preds_a, preds_b, gts, boxes, attrs, img_wh, paths = (
                                                        preds_result["predsA"], 
                                                        preds_result["predsB"], 
                                                        preds_result["gts"], 
                                                        preds_result["boxes"], 
                                                        preds_result["attrs"], 
                                                        preds_result["img_wh"], 
                                                        preds_result["paths"],
                                                    )
occ = attrs[:, 4].astype(bool)

## E-1. 좌표 파라미터화: 예산/지연 여유로 "히트맵이었어야 할 조건" 정량화

In [7]:
n, mb = count_params_mb(LandmarkNet(pretrained=False))
latency_ms = 7.2  # Part B 노트북에서 실측 (Apple M3 Pro, CPU 단일스레드, median)
budget_mb, budget_ms = 25, 20
heatmap_lo, heatmap_hi = latency_ms * 3, latency_ms * 5  # 보수적 3~5배 연산 비용 가정

print(f"예산 여유: 파라미터 {budget_mb/mb:.1f}배, 지연 {budget_ms/latency_ms:.1f}배")
print("직접좌표(현재): GAP->FC 단일 연산이라 저 여유를 거의 그대로 남김.")
print(f"히트맵(64x64x98 + decode) 예상 지연: {heatmap_lo:.1f}~{heatmap_hi:.1f}ms "
      f"(측정된 직접좌표 지연 x 3~5배)")
print(f"-> 현재 예산 {budget_ms}ms로는 부족(하한도 초과). 여유 있게 들어가려면 "
      f"budget이 대략 {heatmap_hi*1.4:.0f}~{heatmap_hi*2:.0f}ms대였어야 함.")

예산 여유: 파라미터 5.6배, 지연 2.8배
직접좌표(현재): GAP->FC 단일 연산이라 저 여유를 거의 그대로 남김.
히트맵(64x64x98 + decode) 예상 지연: 21.6~36.0ms (측정된 직접좌표 지연 x 3~5배)
-> 현재 예산 20ms로는 부족(하한도 초과). 여유 있게 들어가려면 budget이 대략 50~72ms대였어야 함.


## E-2. 정규화와 지표 조건화: yaw × 3종 정규화 표

In [9]:
yaws = np.array([yaw_from_landmarks(g, wh) for g, wh in zip(gts, img_wh)])
idx, labels = bin_by_yaw(yaws, yaw_bins)

print(f"{'yaw bin':10s} {'inter_ocular(A)':>16s} {'inter_pupil(A)':>16s} {'bbox_diag(A)':>14s}")
norms = {}
for norm in ["inter_ocular", "inter_pupil", "bbox_diag"]:
    n_a, _ = nme_batch(preds_a, gts, norm, boxes)
    norms[norm] = n_a
for b, l in enumerate(labels):
    m = idx == b
    if m.sum() == 0:
        continue
    print(f"{l:10s} {norms['inter_ocular'][m].mean():16.4f} "
          f"{norms['inter_pupil'][m].mean():16.4f} {norms['bbox_diag'][m].mean():14.4f}")

lo, hi = norms["inter_ocular"][idx == 0].mean(), norms["inter_ocular"][idx == len(labels)-1].mean()
lo_b, hi_b = norms["bbox_diag"][idx == 0].mean(), norms["bbox_diag"][idx == len(labels)-1].mean()
print(f"\ninter-ocular: 0-10도 -> 45-90도 {hi/lo:.1f}배 증가")
print(f"bbox-diag  : 0-10도 -> 45-90도 {hi_b/lo_b:.1f}배 증가 (더 완만)")

yaw bin     inter_ocular(A)   inter_pupil(A)   bbox_diag(A)
0-10                 0.1654           0.2368         0.0629
10-20                0.1786           0.2548         0.0691
20-30                0.2174           0.3094         0.0799
30-45                0.2910           0.4220         0.0962
45-90                0.4671           0.6871         0.1265

inter-ocular: 0-10도 -> 45-90도 2.8배 증가
bbox-diag  : 0-10도 -> 45-90도 2.0배 증가 (더 완만)


## E-3. confidence와 오차: disagreement vs 실제 NME_B

In [10]:
nme_b, _ = nme_batch(preds_b, gts, "inter_ocular", boxes)
sig = disagreement_signal(preds_a, preds_b)
r = np.corrcoef(sig, nme_b)[0, 1]

low_sig, high_err = sig <= np.median(sig), nme_b > 0.05
print(f"corr(disagreement, NME_B) = {r:.3f}  (양의 상관은 있지만 약함)")
print(f"방향①(저신호·고오차, false-accept) 전체 질량 = {(low_sig & high_err).mean():.3f}"
      f", 신호 하위50% 조건부 = {high_err[low_sig].mean():.3f}")
print(f"방향②(고신호·저오차, false-reject) 전체 질량 = {(~low_sig & ~high_err).mean():.3f}")
print("-> 방향①이 방향②보다 커서, 이 신호는 '몰래 통과'(위험) 쪽으로 치우침.")
print("   Part D에서 disagreement 단독으로 95% precision에 못 미치는(최대 80%) 이유.")

corr(disagreement, NME_B) = 0.484  (양의 상관은 있지만 약함)
방향①(저신호·고오차, false-accept) 전체 질량 = 0.231, 신호 하위50% 조건부 = 0.462
방향②(고신호·저오차, false-reject) 전체 질량 = 0.064
-> 방향①이 방향②보다 커서, 이 신호는 '몰래 통과'(위험) 쪽으로 치우침.
   Part D에서 disagreement 단독으로 95% precision에 못 미치는(최대 80%) 이유.


## E-4. 학습 분포: 어느 subset 결과를 신뢰할지 판정

코드 없이 실측 NME(Part B 노트북)와 데이터 구조 차이로 판단한 표
(전체 서술·근거는 `DESIGN.md` §6/E-4):

| subset | B NME | 전이 신뢰도 | 근거 |
|---|---|---|---|
| pose | 0.161 | 높음 | yaw는 순수 기하 문제(카메라 각도 무관) |
| occlusion | 0.095 | 조건부 | 손 가림은 유사하나 occluder 형태(휠/안전벨트)는 다름 |
| illumination | 0.076 | 낮음 | 웹=RGB 저조도, in-cabin=IR 그레이스케일(센서 모달리티 자체가 다름) |
| blur | 0.087 | 낮음 | 모션/포커스 블러 특성이 다름 |
| make-up | 0.080 | 거의 무관 | in-cabin 대응 개념 자체가 희박 |
| expression | 0.087 | 중간 | 도메인 무관이나 샘플 수가 적어 추정 자체가 불안정 |

함의: 웹 benchmark의 occlusion 강건성은 배포 강건성의 **상한**이지 보장은 아님.